# 🗂️ Segmentación Categórica con K-Modes
> **Caso de negocio:** ICPNA · Perfilamiento de prospectos
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

ICPNA capta prospectos para sus cursos de inglés a través de múltiples canales digitales. La información disponible de cada prospecto es **mayormente categórica** (canal de adquisición, contenido de interés, frecuencia de visita, región, grupo de edad, dispositivo) — variables donde K-Means no aplica porque no hay una noción de "distancia" numérica.

**Objetivo:** Agrupar prospectos en perfiles homogéneos según sus atributos categóricos, para diseñar mensajes y canales de campaña diferenciados por perfil.

**KPIs de éxito:**
- Reducción del costo de K-Modes estable a partir de k=3
- Perfiles interpretables: cada cluster debe tener un patrón categórico dominante claro
- Al menos 3 perfiles accionables para el equipo de marketing

**Algoritmo:** K-Modes (análogo a K-Means para variables categóricas, usa la moda en lugar de la media)

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q kmodes plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kmodes.kmodes import KModes

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Aprendizaje no supervisado — clustering categórico',
    'features': [
        'canal_adquisicion   → canal por el que llegó el prospecto (Google Ads, Facebook, Email, Orgánico, Referido)',
        'categoria_preferida → tema de contenido de mayor interés (Tecnología, Moda, Deportes, Hogar, Libros)',
        'frecuencia_visita   → frecuencia de visita al sitio (Diaria, Semanal, Mensual, Esporádica)',
        'region              → región del prospecto (Lima, Arequipa, Trujillo, Cusco, Otra)',
        'edad_grupo          → rango de edad (18-24, 25-34, 35-44, 45-54, 55+)',
        'dispositivo         → dispositivo principal (Móvil, Desktop, Tablet)',
    ],
    'criterios_aceptacion': {
        'n_clusters': '3 perfiles interpretables',
        'estabilidad_costo': 'costo decrece y se estabiliza a partir de k=3',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con 3 perfiles categóricos naturales
# En producción: reemplazar con datos de formularios + GA4 (BigQuery)
N = 1200

perfiles_param = {
    'digital_joven': {
        'canal':  (['Google Ads', 'Facebook'], [0.55, 0.45]),
        'cat':    (['Tecnologia', 'Moda', 'Deportes'], [0.50, 0.30, 0.20]),
        'freq':   (['Diaria', 'Semanal'], [0.35, 0.65]),
        'region': (['Lima', 'Otra'], [0.80, 0.20]),
        'edad':   (['18-24', '25-34'], [0.40, 0.60]),
        'device': (['Movil', 'Tablet'], [0.85, 0.15]),
    },
    'adulto_tradicional': {
        'canal':  (['Email', 'Organico', 'Referido'], [0.45, 0.35, 0.20]),
        'cat':    (['Hogar', 'Libros', 'Tecnologia'], [0.50, 0.35, 0.15]),
        'freq':   (['Mensual', 'Semanal'], [0.65, 0.35]),
        'region': (['Lima', 'Arequipa', 'Otra'], [0.55, 0.25, 0.20]),
        'edad':   (['35-44', '45-54', '55+'], [0.40, 0.35, 0.25]),
        'device': (['Desktop', 'Movil'], [0.70, 0.30]),
    },
    'regional_esporadico': {
        'canal':  (['Organico', 'Referido', 'Facebook'], [0.40, 0.35, 0.25]),
        'cat':    (['Deportes', 'Hogar', 'Moda'], [0.45, 0.35, 0.20]),
        'freq':   (['Esporadica', 'Mensual'], [0.60, 0.40]),
        'region': (['Arequipa', 'Trujillo', 'Cusco', 'Otra'], [0.30, 0.30, 0.20, 0.20]),
        'edad':   (['25-34', '35-44'], [0.50, 0.50]),
        'device': (['Movil', 'Desktop'], [0.65, 0.35]),
    },
}

rows = []
for i in range(N):
    p = np.random.choice(list(perfiles_param.keys()), p=[0.35, 0.35, 0.30])
    pm = perfiles_param[p]
    rows.append({
        'prospecto_id':         i + 1,
        'canal_adquisicion':    np.random.choice(pm['canal'][0], p=pm['canal'][1]),
        'categoria_preferida':  np.random.choice(pm['cat'][0], p=pm['cat'][1]),
        'frecuencia_visita':    np.random.choice(pm['freq'][0], p=pm['freq'][1]),
        'region':               np.random.choice(pm['region'][0], p=pm['region'][1]),
        'edad_grupo':           np.random.choice(pm['edad'][0], p=pm['edad'][1]),
        'dispositivo':          np.random.choice(pm['device'][0], p=pm['device'][1]),
        'perfil_real':          p,
    })

df = pd.DataFrame(rows)

print(f'Dataset: {df.shape}')
df.head()

In [ ]:
# Distribución de cada variable categórica
FEATURES = ['canal_adquisicion', 'categoria_preferida', 'frecuencia_visita', 'region', 'edad_grupo', 'dispositivo']

fig = make_subplots(rows=2, cols=3, subplot_titles=FEATURES)
for idx, feat in enumerate(FEATURES):
    r, c = idx // 3 + 1, idx % 3 + 1
    counts = df[feat].value_counts()
    fig.add_trace(go.Bar(x=counts.index, y=counts.values, marker_color='#1a4c8c', showlegend=False), row=r, col=c)

fig.update_layout(height=550, title='Distribución de variables categóricas',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Relación canal de adquisición x categoría preferida
cross = pd.crosstab(df['canal_adquisicion'], df['categoria_preferida'], normalize='index')

fig = px.imshow(cross, text_auto='.0%', color_continuous_scale='teal',
                title='Categoría preferida por canal de adquisición (% por fila)')
fig.update_layout(height=420, paper_bgcolor='white')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
X = df[FEATURES].fillna('Desconocido').values

print(f'Observaciones: {X.shape[0]} | Variables categóricas: {X.shape[1]}')
print(f'Valores únicos por variable:')
for f in FEATURES:
    print(f'  {f:20s}: {df[f].nunique()} valores → {sorted(df[f].unique())}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Curva de costo: probar k=2..6
elbow = []
for k in range(2, 7):
    km_tmp = KModes(n_clusters=k, init='Huang', n_init=5, random_state=42)
    km_tmp.fit(X)
    elbow.append({'k': k, 'cost': km_tmp.cost_})

elbow_df = pd.DataFrame(elbow)

fig = go.Figure(go.Scatter(x=elbow_df['k'], y=elbow_df['cost'],
                            mode='lines+markers',
                            line=dict(color='#1a4c8c', width=2.5),
                            marker=dict(size=8, color='#1a4c8c')))
fig.update_layout(title='Curva de costo K-Modes vs número de clusters',
                  xaxis_title='Número de clusters (k)', yaxis_title='Costo (disimilitud total)',
                  height=350, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(dtick=1, gridcolor='#f0f0f0'), yaxis=dict(gridcolor='#f0f0f0'))
fig.show()

In [ ]:
# Entrenar K-Modes con k=3 perfiles
N_CLUSTERS = 3
km = KModes(n_clusters=N_CLUSTERS, init='Huang', n_init=10, random_state=42)
labels = km.fit_predict(X)

df['cluster'] = labels
df['cluster_label'] = [f'Perfil {l+1}' for l in labels]

print(f'Costo final: {km.cost_:.2f}')
print(f'\nTamaño de clusters:')
print(df['cluster_label'].value_counts())

## 5️⃣ Fase 5 — Evaluation

In [ ]:
metrics = {'cost': km.cost_, 'n_clusters': N_CLUSTERS}
print('=== MÉTRICAS DEL MODELO ===')
for k, v in metrics.items():
    print(f'  {k:12s}: {v}')

# Modas (perfil dominante) por cluster
modes_df = pd.DataFrame(km.cluster_centroids_, columns=FEATURES)
modes_df.insert(0, 'cluster', [f'Perfil {i+1}' for i in range(N_CLUSTERS)])
modes_df['n'] = pd.Series(labels).value_counts().sort_index().values

display(modes_df)

In [ ]:
# Heatmap: valor dominante (moda) por cluster y variable, con su frecuencia relativa
clusters = sorted(df['cluster_label'].unique())

profiles_detail = {}
for feat in FEATURES:
    profiles_detail[feat] = {}
    for cl in clusters:
        vc = df[df['cluster_label'] == cl][feat].value_counts(normalize=True)
        profiles_detail[feat][cl] = {vc.index[0]: vc.iloc[0]}
rows_z, text_vals = [], []
for feat in FEATURES:
    row, txt = [], []
    for cl in clusters:
        top = profiles_detail[feat].get(cl, {})
        top_val = max(top, key=top.get) if top else '—'
        top_pct = top.get(top_val, 0) if top else 0
        row.append(top_pct)
        txt.append(f'{top_val}<br>{top_pct:.0%}')
    rows_z.append(row)
    text_vals.append(txt)

fig = go.Figure(go.Heatmap(z=rows_z, x=clusters, y=FEATURES,
                            text=text_vals, texttemplate='%{text}',
                            colorscale='teal', showscale=False))
fig.update_layout(title='Valor dominante por cluster y variable (K-Modes)',
                  height=400, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Recomendación de campaña por perfil
recomendaciones = {
    'Perfil 1': '',
    'Perfil 2': '',
    'Perfil 3': '',
}
for cl in clusters:
    canal_dom = profiles_detail['canal_adquisicion'][cl]
    dispositivo_dom = profiles_detail['dispositivo'][cl]
    categoria_dom = profiles_detail['categoria_preferida'][cl]
    canal_top = max(canal_dom, key=canal_dom.get)
    dispositivo_top = max(dispositivo_dom, key=dispositivo_dom.get)
    categoria_top = max(categoria_dom, key=categoria_dom.get)
    recomendaciones[cl] = (f'Priorizar {canal_top} con creativos mobile-first ({dispositivo_top}), '
                           f'temática "{categoria_top}"')

for cl, rec in recomendaciones.items():
    n = (df['cluster_label'] == cl).sum()
    print(f'{cl} ({n} prospectos, {n/len(df):.0%}):')
    print(f'  → {rec}\n')

In [ ]:
# Guardar outputs
df[['prospecto_id'] + FEATURES + ['cluster_label']].to_csv('prospectos_segmentados_icpna.csv', index=False)
print('Archivo prospectos_segmentados_icpna.csv guardado ✓')

import joblib
joblib.dump({'model': km, 'features': FEATURES, 'cluster_modes': modes_df},
            'modelo_kmodes_icpna.pkl')
print('Modelo modelo_kmodes_icpna.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
n_prospectos = len(df)

print('=== RESUMEN EJECUTIVO ===')
print(f'Prospectos segmentados:             {n_prospectos:,}')
print(f'Perfiles identificados:              {N_CLUSTERS}')
for cl in clusters:
    n = (df['cluster_label'] == cl).sum()
    print(f'  {cl}: {n:,} prospectos ({n/n_prospectos:.0%})')
print(f'Costo final K-Modes:                 {km.cost_:.1f}')
print(f'\nArquitectura de producción:')
print('  Formularios + GA4 → BigQuery → recalculo trimestral de perfiles → segmentación de campañas (Google/Meta Ads)')